# American Express – Data Engineer Interview Questions


**Position:** Data Engineer


## Round 1 – Technical

**1.** How would you design a data platform to handle real-time credit card transaction data for fraud analytics?

**2.** Explain your approach to ensuring **ACID compliance** when ingesting streaming data into a data warehouse.

**3.** You’re given a large volume of semi-structured data (**JSON/Avro**) – how do you design an optimal storage & query strategy?

**4.** A team introduces a new upstream API with changed response fields. How do you version-control and evolve your schema?

**5.** How would you implement data lineage and auditing for sensitive financial data pipelines?



## 1.


> **“For real-time credit card fraud analytics, I would design a streaming data platform where transactions are ingested with very low latency, processed for fraud features and rules, scored by a fraud model, and then stored for both real-time decisions and historical analytics.”**

### 1. Data Ingestion

* Credit-card transactions would come from payment systems, APIs, or transaction databases.
* I would publish events to **Kafka/Event Hubs** using a unique `transaction_id`.
* Use multiple partitions based on a suitable key such as `card_id` or `customer_id` to achieve parallel processing.
* Enable replication and acknowledgements to avoid data loss.

### 2. Real-Time Processing

* Use **Spark Structured Streaming** or another stream-processing engine.
* Validate the schema and perform:

  * Null/format validation
  * Duplicate detection
  * Data cleansing
  * Enrichment with customer, merchant, and location information.
* Use **event time + watermarks** to handle late-arriving transactions.

### 3. Fraud Detection

I would calculate real-time features such as:

* Number of transactions in the last 5 minutes
* Transaction amount compared with historical average
* Multiple transactions from different locations within a short period
* Unusual merchant/category
* Failed transaction frequency
* Device/IP/location changes

Then apply **rule-based checks + ML fraud scoring**.

For example:

```text
Transaction
     ↓
Kafka / Event Hubs
     ↓
Spark Structured Streaming
     ↓
Validation + Deduplication
     ↓
Enrichment + Real-Time Features
     ↓
Fraud Rules + ML Model
     ↓
Fraud Score
     ↓
 ┌───────────────┬────────────────┐
 ↓               ↓                ↓
Approve       Review/Alert      Block
```

### 4. Storage Architecture

I would use a **Bronze → Silver → Gold** architecture:

* **Bronze:** Raw transaction events in Delta Lake/ADLS.
* **Silver:** Cleaned, deduplicated and enriched transactions.
* **Gold:** Fraud features, fraud scores, aggregated customer/merchant metrics.

For historical analytics, I would use **Databricks/Delta Lake or a cloud data warehouse**.

### 5. Reliability & Data Quality

I would implement:

* Checkpointing for streaming jobs
* Exactly-once processing where supported
* Idempotent writes using `transaction_id`
* Retry and dead-letter queues for bad events
* Schema validation and schema evolution
* Data-quality checks and reconciliation
* Monitoring for throughput, latency, failures and dropped records

### 6. Security

Since this is financial data, I would apply:

* Encryption in transit and at rest
* RBAC/least-privilege access
* Tokenization/masking of card and PII data
* Secrets managed through a secure secret store
* Audit logging
* Data retention and governance policies

### 7. Monitoring & Alerting

I would monitor:

* Kafka/Event Hub lag
* Streaming processing latency
* Failed transactions
* Data-quality failures
* Fraud-model latency
* End-to-end SLA

Alerts can be integrated with **Azure Monitor/Log Analytics/PagerDuty** depending on the environment.

### Interview Closing Statement

> **“The key design goals would be low-latency fraud detection, no transaction loss or duplication, scalable processing, strong security for PCI/PII data, and complete auditability. I would also separate the real-time fraud decision path from the historical analytics path so that heavy analytical workloads don't impact transaction-processing SLAs.”**

**One-line architecture:**
**Transaction Sources → Kafka/Event Hubs → Spark Structured Streaming → Validation/Dedup → Enrichment & Feature Engineering → Fraud Rules + ML Model → Real-Time Decision/Alert → Delta Lake/Warehouse → BI & Analytics**.


## 2.


> **“When ingesting streaming data into a data warehouse, I focus on atomic writes, consistency, duplicate prevention, isolation, and recovery. I would typically use a transactional storage layer such as Delta Lake before loading the warehouse.”**

### 1. Ingest the stream reliably

* Consume events using **Kafka/Event Hubs + Spark Structured Streaming**.
* Use **checkpointing** to track processed offsets.
* Use a unique business key such as `transaction_id`.

### 2. Ensure Atomicity

* Write each micro-batch as a **single transaction**.
* If any operation fails, the entire transaction is rolled back rather than partially committing.
* With Delta Lake, the transaction log provides atomic commits.

### 3. Prevent Duplicates

Streaming systems can potentially reprocess messages during failures.

I would implement **idempotent processing**:

```text
transaction_id = unique key
        ↓
Check existing record
        ↓
New → INSERT
Existing → UPDATE/IGNORE
```

For example, use a Delta `MERGE`:

```sql
MERGE INTO target t
USING source s
ON t.transaction_id = s.transaction_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *;
```

This helps ensure that retrying the same micro-batch doesn't create duplicate transactions.

### 4. Maintain Consistency

Before committing data, perform:

* Schema validation
* Data-type validation
* Null checks
* Business-rule validation
* Referential/data-quality checks

Invalid records can be moved to a **dead-letter/quarantine table** instead of corrupting the target dataset.

### 5. Isolation

* Use the transactional capabilities of **Delta Lake or the target warehouse**.
* Readers should see a consistent committed version rather than partially written data.
* Concurrent writes are controlled through transaction/version mechanisms.

### 6. Durability

Once the transaction commits:

* Persist data in durable cloud storage.
* Enable replication where appropriate.
* Maintain transaction logs/checkpoints.
* Configure appropriate backup/retention policies.

### 7. Failure & Recovery

```text
Kafka/Event Hub
      ↓
Spark Structured Streaming
      ↓
Checkpoint + Offset Tracking
      ↓
Validation / Deduplication
      ↓
Delta Lake
      ↓
Warehouse
```

If the Spark job fails:

* Restart from the **last successful checkpoint**.
* Reprocess the required events.
* Use idempotent `MERGE`/deduplication so retries don't create duplicates.

### Interview Closing Answer

> **“So my approach is to combine checkpointing for reliable stream processing, transactional storage such as Delta Lake for ACID guarantees, unique transaction IDs and MERGE logic for idempotency, validation for consistency, and proper recovery mechanisms for failures. This gives me atomic, consistent, isolated and durable streaming ingestion while preventing duplicates and partial writes.”**

**Key keywords to mention in the interview:**
**Checkpointing → Exactly-once semantics → Idempotency → MERGE → Delta transaction log → Atomic commits → Schema validation → Recovery.**


## 3.


> **“For large volumes of JSON/Avro data, I would separate ingestion format from the analytics storage format. I would retain the raw data for traceability, then convert it into a columnar format such as Parquet/Delta for efficient querying.”**

### 1. Storage Architecture

I would use a **Bronze → Silver → Gold** approach:

```text
JSON / Avro Sources
        ↓
   ADLS / S3
        ↓
 Bronze – Raw Data
        ↓
 Schema Validation
        ↓
 Silver – Parquet / Delta
        ↓
 Transformation & Aggregation
        ↓
 Gold – Analytics Tables
```

* **Bronze:** Preserve original JSON/Avro for replay and auditing.
* **Silver:** Flatten/normalize frequently queried fields and store as **Parquet/Delta**.
* **Gold:** Create business-level aggregates optimized for reporting.

### 2. Use Columnar Storage

I would avoid querying large volumes of raw JSON directly.

Instead:

* Convert JSON/Avro → **Parquet/Delta**
* Store data in compressed columnar format.
* Read only the required columns.
* Benefit from **column pruning and predicate pushdown**.

For example, if analysts only need:

```text
transaction_id
customer_id
amount
transaction_date
```

the engine doesn't need to scan every column from the dataset.

### 3. Partitioning

Partition based on columns commonly used for filtering, typically:

```text
year/month/day
```

For example:

```text
transactions/
 ├── year=2026/
 │    ├── month=09/
 │    │    ├── day=25/
 │    │    └── day=26/
```

I would **avoid over-partitioning**, especially by high-cardinality columns such as `customer_id`, because it can create millions of small files.

### 4. Handle Schema Evolution

For JSON/Avro, schemas can change over time.

I would:

* Maintain a schema registry/versioning mechanism.
* Validate incoming schemas.
* Support backward/forward compatibility where appropriate.
* Add new nullable columns safely.
* Quarantine incompatible records instead of failing the entire pipeline.

### 5. Optimize Query Performance

For a Databricks/Delta environment, I would use:

* **OPTIMIZE** to compact small files.
* Appropriate clustering/data skipping techniques.
* Statistics on frequently filtered columns.
* Predicate pushdown.
* Column pruning.
* Avoid unnecessary `SELECT *`.
* Cache only frequently reused datasets.

### 6. Handle Small Files

High-volume streaming ingestion can create many small files.

I would implement:

```text
Streaming ingestion
       ↓
Small files
       ↓
Compaction / OPTIMIZE
       ↓
Larger optimized files
```

This reduces metadata overhead and improves query performance.

### 7. Data Governance & Security

For sensitive data:

* Encrypt data at rest and in transit.
* Apply RBAC/ACLs.
* Mask/tokenize PII.
* Maintain data lineage and audit logs.
* Apply appropriate retention policies.

### Interview Closing Answer

> **“So, I would keep the original JSON/Avro in the Bronze layer for audit and replay, transform it into Parquet/Delta in the Silver layer for efficient analytical access, and create optimized Gold tables for business queries. I would use sensible date-based partitioning, compression, predicate pushdown, column pruning and file compaction, while using schema versioning to handle schema evolution. The main goal is to minimize storage and query cost without sacrificing data traceability and scalability.”**

**Key keywords to mention:**
**JSON/Avro → Bronze → Parquet/Delta → Partitioning → Compression → Predicate Pushdown → Column Pruning → Schema Evolution → Small-file Compaction → Data Governance.**


## 4.


> **“I would treat the API schema as a versioned contract between the upstream and downstream systems. Before accepting the change, I would identify whether it is backward-compatible or a breaking change, then manage the schema accordingly.”**

### 1. Detect & Validate the Schema Change

First, compare the new API response against the existing schema.

Examples:

* **Compatible:** Adding a new optional field.
* **Potentially breaking:** Changing a field's data type.
* **Breaking:** Removing or renaming an existing field.

I would validate incoming payloads against the expected schema and quarantine invalid records.

### 2. Maintain Schema Versions

I would maintain explicit versions, for example:

```text
API Schema v1
  customer_id
  transaction_id
  amount

API Schema v2
  customer_id
  transaction_id
  transaction_amount
  currency
```

Use a **Schema Registry** or version-controlled schema definitions in Git.

```text
Git / Schema Registry
        ↓
     Schema v1
     Schema v2
     Schema v3
```

### 3. Handle Backward Compatibility

For non-breaking changes, I would evolve the existing schema.

For example, if `currency` is newly introduced:

```text
transaction_id
amount
currency → nullable
```

Existing records can continue to work because the new field has a default/null value.

### 4. Handle Breaking Changes

If the API changes:

```text
amount → transaction_amount
```

I would **not immediately overwrite the existing contract**.

Instead:

```text
API v1 → Existing Pipeline → Existing Schema
API v2 → New Schema/Transformation
                     ↓
              Canonical Model
```

The transformation layer maps both versions into a common downstream structure.

### 5. Version the Data Pipeline

I would maintain separate transformations where required:

```text
API v1 ──→ Parser v1 ──┐
                       ├──→ Canonical Schema → Silver → Gold
API v2 ──→ Parser v2 ──┘
```

This allows downstream consumers to migrate without breaking existing workloads.

### 6. Testing Before Production

Before deploying the new schema, I would perform:

* Contract/schema validation
* Backward-compatibility testing
* Data-quality checks
* Integration testing
* Regression testing
* Sample-data reconciliation

I would also run **v1 and v2 in parallel** for a period if the change is significant.

### 7. Monitoring & Rollback

I would monitor:

* Schema validation failures
* Missing/new fields
* Null-rate changes
* Data-type mismatches
* Record counts
* Source-to-target reconciliation

If the new version causes issues, I should be able to **roll back to the previous pipeline/schema version**.

### Interview Closing Answer

> **“My approach is to treat schema as a contract. I first classify the API change as compatible or breaking, maintain schemas in Git or a Schema Registry, and use explicit versions. For compatible changes, I evolve the existing schema with optional/default fields. For breaking changes, I introduce a new version and transform both versions into a canonical downstream model. I would test the new contract, run versions in parallel when necessary, monitor data quality, and have a rollback strategy. This allows the upstream API to evolve without unexpectedly breaking downstream pipelines.”**

**Key keywords:**
**Schema Contract → Schema Registry → Git Versioning → Backward Compatibility → Schema Evolution → API v1/v2 → Canonical Model → Contract Testing → Parallel Run → Rollback.**


## 5.


> **“For sensitive financial data, I would implement lineage at both the dataset and column level, while maintaining an immutable audit trail of who accessed or modified the data. My goal would be to answer three questions: where did the data come from, how was it transformed, and who accessed or changed it?”**

### 1. Capture End-to-End Data Lineage

I would track the complete flow:

```text
Source Systems
     ↓
Kafka / API / DB
     ↓
Bronze
     ↓
Silver
     ↓
Gold
     ↓
Data Warehouse / BI
```

For example:

```text
transaction.amount
        ↓
Bronze.amount
        ↓
Silver.amount
        ↓
Gold.daily_transaction_amount
        ↓
Power BI Report
```

This allows us to perform **impact analysis** when a source field changes.

### 2. Use a Data Catalog

I would integrate a governance/catalog solution such as **Microsoft Purview** in Azure.

It can maintain:

* Dataset/table lineage
* Column-level lineage
* Data ownership
* Data classification
* Business glossary
* Source-to-target relationships

For a Databricks environment, I would also use **Unity Catalog** for centralized governance and access control.

### 3. Implement Audit Logging

Every important operation should generate an audit record containing information such as:

```text
timestamp
user/service_principal
source_system
dataset/table
operation
pipeline/job
records_processed
status
correlation_id
```

For example:

```text
2026-09-26 19:00
Pipeline: Transaction_Ingestion
User: service-principal-ADF
Source: Payment API
Target: Silver.Transactions
Records: 2.4M
Status: SUCCESS
```

### 4. Protect Sensitive/PII Data

For financial data, I would classify sensitive fields such as:

* Card number
* Customer ID
* Account information
* Personal information

Then apply:

* Encryption at rest and in transit
* RBAC / least privilege
* Dynamic masking where appropriate
* Tokenization
* Restricted access to raw/PII zones
* Secrets stored in **Azure Key Vault**

### 5. Data Quality & Reconciliation

I would maintain audit/control tables to capture:

```text
source_count
target_count
rejected_count
duplicate_count
processing_start_time
processing_end_time
pipeline_status
```

For example:

```text
Source records = 1,000,000
Target records =   998,500
Rejected       =     1,500
```

The pipeline should flag the batch if the reconciliation rules fail.

### 6. Immutable Audit Trail

For financial systems, audit records should be protected from unauthorized modification.

I would:

* Restrict write/delete permissions.
* Store audit logs in secure centralized storage.
* Apply retention policies.
* Monitor suspicious access.
* Integrate logs with **Azure Monitor / Log Analytics / SIEM** where appropriate.

### 7. Monitoring & Alerting

I would create alerts for:

* Unauthorized data access
* Failed pipelines
* Unexpected record-count changes
* Schema changes
* PII exposure
* Permission changes
* Repeated authentication failures

### Interview Closing Answer

> **“So my design would combine Purview or Unity Catalog for data lineage and governance, centralized audit logging for user and pipeline activity, RBAC and encryption for sensitive data, and reconciliation tables for data-level auditing. Every pipeline execution would have a correlation ID so I can trace a transaction from the source through Bronze, Silver and Gold to the final report. This gives us end-to-end traceability, security, compliance and faster root-cause analysis.”**

### Key keywords to mention

**Data Lineage → Column-Level Lineage → Microsoft Purview → Unity Catalog → Audit Logs → RBAC → PII Masking → Encryption → Key Vault → Reconciliation → Correlation ID → Immutable Audit Trail → SIEM/Monitoring.**


## -------------------------------------------------------------------------------- 

## Round 2 – Advanced Technical

**6.** Your ETL pipeline is intermittently dropping records. Walk me through your debugging process.

**7.** How do you secure **PII data** in a multi-tenant data lake but still enable analytics?

**8.** Explain your approach to optimizing **Spark jobs** to reduce shuffle & skew at scale.

**9.** A downstream model needs strict SLAs. How would you design a reliable alerting & recovery mechanism?

**10.** How would you partition and index **petabyte-scale data** for high-performance reporting?



## 6.


> **“If an ETL pipeline is intermittently dropping records, I would first identify exactly where the records are being lost, then isolate whether the issue is at ingestion, transformation, or target-write level. I would use record counts, audit IDs, logs, and correlation IDs to trace the affected records end-to-end.”**

### 1. Establish Source-to-Target Reconciliation

First, I would compare counts at every stage:

```text
Source
  ↓
Ingestion
  ↓
Raw/Bronze
  ↓
Transformation
  ↓
Silver
  ↓
Target/Warehouse
```

For example:

```text
Source        = 10,000
Bronze        = 10,000
After filter  =  9,850
Silver        =  9,850
Target        =  9,800
```

This immediately tells me **where the 50 records disappeared**.

### 2. Trace Individual Records

I would identify a unique key such as:

```text
transaction_id / order_id / customer_id
```

Then trace missing records through every stage.

I'd also use a **correlation/batch ID** to identify which pipeline execution produced the data.

### 3. Check the Ingestion Layer

If records are missing before transformation, I would investigate:

* Kafka/Event Hub offsets
* API pagination
* CDC configuration
* Source connectivity
* Network timeouts
* Retry behavior
* Checkpoint management
* Message acknowledgement
* Duplicate/offset handling

For APIs, I would specifically verify whether pagination or rate limits are causing partial extraction.

### 4. Check Transformation Logic

Next, I would inspect:

* `INNER JOIN` unintentionally removing records
* Incorrect `WHERE` filters
* Null handling
* Deduplication logic
* Incorrect window-function logic
* Data-type conversion failures
* Schema mismatch
* Late-arriving data handling

A common example is:

```sql
FROM transactions t
INNER JOIN customers c
  ON t.customer_id = c.customer_id
```

If a customer doesn't exist in the customer table, the transaction disappears. I would test this by temporarily using a `LEFT JOIN` and identifying unmatched records.

### 5. Check Target Write

If the records exist in Silver but not in the target, I would investigate:

* Failed/partial writes
* `MERGE` conditions
* Primary-key conflicts
* Transaction rollbacks
* Warehouse constraints
* Partition problems
* Concurrent writes
* Batch failures/retries

### 6. Check Logs & Metrics

I would correlate:

```text
Pipeline Run ID
       ↓
Source Count
       ↓
Read Count
       ↓
Filtered Count
       ↓
Rejected Count
       ↓
Written Count
       ↓
Target Count
```

I'd review ADF/Spark/Databricks logs and metrics around the exact time the issue occurred.

### 7. Reproduce the Failure

Since the problem is **intermittent**, I would compare:

* Successful vs failed pipeline runs
* Data volume
* Source partitions
* Specific time windows
* Specific partitions/files
* Schema versions
* Cluster/resource conditions

This helps determine whether it's a **data-dependent or infrastructure-dependent** issue.

### 8. Fix & Prevent Recurrence

Once the root cause is identified, I would implement:

* Idempotent processing
* Checkpointing
* Retry with exponential backoff
* Dead-letter/quarantine handling
* Source-to-target reconciliation
* Automated data-quality checks
* Alerts when count thresholds fail
* Audit/control tables

For example:

```text
Source Count = Read Count + Rejected Count
Target Count = Expected Valid Count
```

If reconciliation fails, the pipeline should **fail/alert rather than silently succeed**.

### Interview Closing Answer

> **“My debugging approach is to first reconcile record counts across every pipeline layer, then trace specific missing records using business keys and correlation IDs. I would check ingestion offsets and retries, transformation filters and joins, deduplication logic, and finally target-write or merge behavior. Since the issue is intermittent, I would compare successful and failed runs to identify a common condition. After fixing the root cause, I would add reconciliation, idempotency, quarantine handling, and monitoring so that records cannot be silently dropped again.”**

**Key keywords:**
**Source-to-Target Reconciliation → Correlation ID → Audit Tables → Logs → CDC/Offsets → Join/Filter Validation → Deduplication → MERGE → Checkpointing → Idempotency → Data Quality Alerts.**


## 7.

A strong **interview-type answer** would be:

> **“For PII in a multi-tenant data lake, I would follow a defense-in-depth approach: isolate tenants, minimize exposure of PII, enforce fine-grained access control, encrypt sensitive data, and provide analysts with masked or tokenized data instead of raw PII.”**

### 1. Tenant Isolation

I would logically separate tenant data using:

```text id="7jyv5a"
Data Lake
   │
   ├── Tenant A
   ├── Tenant B
   └── Tenant C
```

Depending on requirements, this could be implemented using separate containers/storage paths, catalogs/schemas, or tables with a `tenant_id`.

Access policies would ensure that users from **Tenant A cannot access Tenant B's data**.

### 2. Data Classification

First, identify and classify sensitive fields:

```text id="8jlvjt"
PII:
- Name
- Email
- Phone
- Address
- Customer ID

Highly sensitive:
- Card/Account information
- Government identifiers
```

Only the fields actually required for analytics should be exposed.

### 3. Masking & Tokenization

I would avoid exposing raw PII to analysts.

For example:

```text id="u0d0xv"
Original Email:
pritam@example.com

Analytics View:
p*****@example.com
```

For fields that need consistent joins, I would use **tokenization or deterministic hashing**:

```text id="f2gq4v"
customer_id
     ↓
Tokenization
     ↓
TOKEN_8A72F1
```

This allows analysts to join and aggregate records without seeing the actual identifier.

### 4. Fine-Grained Access Control

Use **RBAC + ABAC/row-level/column-level security** where supported.

For example:

```text id="t5w8tb"
Data Engineer → Operational data
Analyst       → Masked PII
Compliance    → Controlled PII access
Admin         → Full access, audited
```

In Azure/Databricks, I would consider **Microsoft Purview, Unity Catalog, Azure RBAC and storage ACLs** as appropriate to the architecture.

### 5. Encryption

Use encryption:

* **At rest:** ADLS/Delta/storage encryption
* **In transit:** TLS
* Customer-managed keys where regulatory requirements justify them
* Store secrets and keys securely, e.g. **Azure Key Vault**

### 6. Analytics Without Raw PII

I would create curated analytics views:

```text id="a1qjkv"
Raw PII Data
     ↓
Restricted Zone
     ↓
Tokenization / Masking
     ↓
Analytics View
     ↓
BI / ML / Reporting
```

For example, analysts may need:

```text id="1f9f1v"
tenant_id
customer_token
transaction_amount
transaction_date
merchant_category
```

They don't necessarily need the customer's name, phone number, or full card number.

### 7. Auditing & Monitoring

Every sensitive-data access should be logged:

```text id="b1o3wm"
User
Tenant
Dataset
Columns accessed
Operation
Timestamp
Success/Failure
```

I would configure alerts for unusual access patterns and periodically review permissions.

### 8. Data Retention & Governance

I would also implement:

* Data retention policies
* Secure deletion
* Data classification
* Data lineage
* Access reviews
* Least-privilege permissions
* Separation of duties

### Interview Closing Answer

> **“My approach is to keep raw PII in a highly restricted zone, isolate tenants through storage/catalog and access policies, encrypt the data, and expose analysts primarily to masked or tokenized datasets. Fine-grained RBAC and row/column-level controls prevent unauthorized access, while audit logs provide traceability. This gives analysts the data they need for aggregation, joins and ML without unnecessarily exposing sensitive customer information.”**

**Key keywords:**
**Tenant Isolation → Data Classification → PII Masking → Tokenization → RBAC → Row/Column-Level Security → Encryption → Key Vault → Audit Logging → Data Minimization → Retention → Least Privilege.**


## 8.



> **“At scale, I optimize Spark jobs by first identifying whether the bottleneck is caused by excessive shuffle, data skew, poor partitioning, or inefficient joins. I use the Spark UI and execution plan to identify the expensive stages before making changes.”**

### 1. Reduce Unnecessary Shuffle

Shuffle is usually expensive because data moves across executors.

I would:

* Avoid unnecessary `repartition()`.
* Use `coalesce()` when reducing partitions after filtering.
* Filter data **before joins and aggregations**.
* Select only required columns before shuffle.
* Avoid unnecessary `distinct()` and `groupBy()`.
* Pre-aggregate data where possible.

```python
df = (
    df.filter("transaction_date >= '2026-01-01'")
      .select("customer_id", "amount")
)
```

### 2. Optimize Joins

For a small lookup table, I would use a **broadcast join**:

```python
from pyspark.sql.functions import broadcast

result = transactions.join(
    broadcast(customers),
    "customer_id",
    "left"
)
```

This avoids a large shuffle of the transaction dataset.

For large-to-large joins, I would optimize partitioning and consider **AQE**.

### 3. Handle Data Skew

I first identify skew using the **Spark UI**, partition-size metrics, or key-frequency analysis.

For example:

```text
customer_id = 1001 → 50 million records
customer_id = 1002 → 10,000 records
customer_id = 1003 → 8,000 records
```

One partition can become much larger than the others.

Depending on the scenario, I would use:

* **AQE skew join optimization**
* Salting for heavily skewed keys
* Broadcast join if one side is sufficiently small
* Pre-aggregation
* Better partitioning strategy

### 4. Use Salting for Severe Skew

For a highly skewed key:

```text
Original:
customer_id = 1001 → 50M records
```

I can create a salt value:

```text
customer_id + salt
```

and distribute the large key across multiple partitions.

```python
from pyspark.sql.functions import floor, rand, concat_ws

df = df.withColumn("salt", floor(rand() * 10))
```

The join logic must apply the same compatible salting strategy to the other side.

### 5. Tune Partitioning

I would avoid both:

* **Too few partitions** → under-utilized cluster
* **Too many partitions** → scheduling and shuffle overhead

I would tune:

```text
spark.sql.shuffle.partitions
```

based on **data volume, cluster size, and workload**, rather than blindly setting a fixed number.

### 6. Enable Adaptive Query Execution

For modern Spark workloads, I would enable:

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
```

AQE can dynamically optimize the execution plan based on runtime statistics, including:

* Coalescing small shuffle partitions
* Handling skewed joins
* Optimizing join strategies

### 7. Optimize Data Layout

For large datasets, I would use:

* **Parquet/Delta**
* Partition pruning
* Predicate pushdown
* Column pruning
* File compaction
* Appropriate clustering/data-skipping techniques

I would avoid excessive partitioning, especially on high-cardinality columns.

### 8. Cache Carefully

I would use `cache()`/`persist()` **only when a DataFrame is reused multiple times**.

I wouldn't cache large datasets unnecessarily because it can create memory pressure and potentially cause spills.

### 9. Measure Before & After

I would use:

```text
Spark UI
   ↓
Stages
   ↓
Shuffle Read / Shuffle Write
   ↓
Task Duration
   ↓
Partition Size
   ↓
Spill / GC
```

Then compare metrics before and after optimization.

### Interview Closing Answer

> **“My optimization process is measurement-driven. I first use the Spark UI and execution plan to identify shuffle-heavy stages and skewed partitions. Then I reduce unnecessary shuffles through early filtering, column pruning and pre-aggregation, use broadcast joins for small datasets, handle severe skew with AQE or salting, and tune partition counts based on workload size. Finally, I optimize the storage layout and validate the improvement using shuffle, task-duration, spill and execution-time metrics.”**

### Quick interview checklist

**Spark UI → Identify Shuffle → Filter Early → Column Pruning → Broadcast Join → AQE → Handle Skew → Repartition Carefully → Avoid Small Files → Cache Only When Needed → Measure Again.**


## 9.


> **“If a downstream model has a strict SLA, I would design the pipeline around SLA monitoring, proactive alerting, automatic recovery, and a clear fallback mechanism. The key is to detect an issue before the SLA is breached and recover without producing incomplete or duplicate data.”**

### 1. Define the SLA & SLO

First, I would clearly define:

```text
Data availability SLA → 7:00 AM
Pipeline completion    → < 60 minutes
Data freshness         → < 15 minutes
Accuracy/reconciliation → 99.9%
```

I would establish these as measurable metrics rather than simply monitoring whether the pipeline succeeded.

### 2. End-to-End Monitoring

I would monitor every critical stage:

```text
Source
  ↓
Ingestion
  ↓
Transformation
  ↓
Data Quality
  ↓
Model Features
  ↓
Model
```

Key metrics:

* Pipeline duration
* Data freshness
* Kafka/Event Hub lag
* Record counts
* Processing latency
* Failed tasks
* Data-quality failures
* Model-input availability

### 3. Proactive Alerting

I wouldn't wait until the SLA is already breached.

For example:

```text
SLA = 7:00 AM

6:00 → Normal
6:20 → Warning
6:40 → Critical
7:00 → SLA Breach
```

Alerts can be sent through **Azure Monitor / Log Analytics**, email, Teams, PagerDuty, or another incident-management system.

### 4. Automatic Retry & Recovery

For transient failures, I would implement:

* Retry with exponential backoff
* Configurable retry limits
* Checkpoint-based recovery
* Idempotent processing
* Automatic pipeline restart

For example:

```text
Pipeline Failure
      ↓
Classify Error
      ↓
Transient? ── Yes → Retry
      │
      No
      ↓
Quarantine / Incident
      ↓
Alert On-Call
```

### 5. Checkpointing & Idempotency

For Spark Streaming, I would use **checkpointing** so the job can resume from the last successful state.

I would also make writes **idempotent**, for example using a unique business key and `MERGE`, so a retry doesn't duplicate records.

```text
transaction_id
       ↓
Already processed? → UPDATE/IGNORE
New record?        → INSERT
```

### 6. Data Quality Gates

Before making data available to the model, I would validate:

* Record count
* Null percentage
* Duplicate count
* Schema
* Referential integrity
* Expected data freshness

If critical checks fail, I would **stop downstream model execution** rather than feed bad or incomplete data.

### 7. Backfill & Replay Strategy

I would maintain the ability to replay data from:

* Kafka/Event Hub offsets
* Raw Bronze layer
* Checkpoints
* Historical snapshots

This allows us to recover missing data without manually reconstructing it.

### 8. Fallback Mechanism

For a very strict SLA, I would design a controlled fallback.

For example:

```text
Latest Dataset
     ↓
Fresh Data Available?
   ↙       ↘
 Yes        No
 ↓           ↓
Run Model   Use Last Valid Snapshot
             +
          Alert Team
```

The fallback should only be used if the business/model requirements permit it.

### 9. Observability & Runbook

Every pipeline execution should have a:

* Run ID
* Correlation ID
* Start/end timestamp
* Source/target counts
* Status
* Error details

I would also maintain an operational **runbook** describing what to do for common failures.

### Interview Closing Answer

> **“My design would combine proactive SLA monitoring, stage-level observability, automated retries, checkpointing, idempotent processing, data-quality gates, and a replay/backfill mechanism. I would introduce warning and critical thresholds before the actual SLA deadline so the team has time to recover. If recovery isn't possible within the SLA window, a controlled fallback such as the last validated dataset can be used when business rules allow it. Most importantly, I would prevent incomplete or corrupted data from reaching the downstream model.”**

### Quick interview flow

**SLA Definition → Monitoring → Early Warning → Automated Retry → Checkpoint Recovery → Data Quality Gate → Replay/Backfill → Controlled Fallback → Incident Alerting → Root-Cause Analysis**.


## 10.


> **“For petabyte-scale reporting, I would avoid relying on traditional database indexing alone. I would design the storage layout around query patterns, use appropriate partitioning and clustering, and minimize the amount of data scanned by every query.”**

### 1. Understand Query Patterns First

Before choosing partitions, I would analyze:

* Most frequently filtered columns
* Common `WHERE` conditions
* Join keys
* Time-based reporting requirements
* Cardinality of columns
* Data growth rate

For example, if most reports are based on transaction date:

```text
transaction_date
        ↓
year → month → day
```

### 2. Use Time-Based Partitioning

For a transaction dataset:

```text
transactions/
├── year=2026/
│   ├── month=09/
│   │   ├── day=25/
│   │   └── day=26/
```

This enables **partition pruning**.

A query such as:

```sql
SELECT *
FROM transactions
WHERE transaction_date >= '2026-09-01'
  AND transaction_date < '2026-10-01';
```

doesn't need to scan the entire petabyte dataset.

### 3. Avoid Over-Partitioning

I wouldn't partition directly by high-cardinality columns such as:

```text
customer_id
transaction_id
```

because this can create an enormous number of small partitions/files.

Instead, I would use a **low/medium-cardinality partition column**, typically date or another strong business dimension.

### 4. Use Clustering / Data Skipping

Within partitions, I would organize data around frequently filtered columns.

For example:

```text
Partition → transaction_date
Cluster/Z-Order → customer_id, merchant_id
```

Depending on the platform, this could use:

* Databricks liquid clustering / data skipping
* Delta optimization techniques
* Snowflake clustering
* Warehouse-specific clustering/indexing mechanisms

The goal is to reduce the amount of data that needs to be scanned.

### 5. Indexing Strategy

At petabyte scale, I wouldn't blindly create traditional indexes on every column.

Instead, I'd use indexes or equivalent structures where supported and where query patterns justify them.

For example:

```text
Frequently filtered:
customer_id
merchant_id
transaction_date

Frequently joined:
customer_id
merchant_id
```

For analytical platforms, **partition pruning + clustering + columnar storage** often provides more benefit than maintaining traditional B-tree indexes everywhere.

### 6. Use Columnar Storage

I would store the data in **Parquet/Delta** rather than row-oriented formats.

Benefits:

* Column pruning
* Compression
* Predicate pushdown
* Reduced I/O
* Efficient aggregation

For example, if a report only needs:

```text
customer_id
transaction_date
amount
```

the engine doesn't need to read every column.

### 7. Handle Small Files

At petabyte scale, file management becomes critical.

I would implement:

* File compaction
* Appropriate target file sizes
* Avoid excessive streaming micro-batches
* Periodic optimization

For Databricks/Delta, this could involve `OPTIMIZE` or appropriate clustering/compaction strategies.

### 8. Create Aggregate Tables

For frequently accessed dashboards, I wouldn't make BI tools scan raw petabytes every time.

I'd create:

```text
Raw Transactions
       ↓
Detailed Fact Table
       ↓
Daily/Monthly Aggregates
       ↓
BI Dashboard
```

For example:

```sql
customer_id
transaction_date
total_amount
transaction_count
```

This can dramatically reduce reporting latency and compute cost.

### 9. Separate Hot & Cold Data

I would also consider data lifecycle management:

```text
Recent Data → High-performance storage
Historical Data → Cost-optimized storage
```

Frequently accessed recent data can remain readily queryable, while older data can use lower-cost storage depending on business and regulatory requirements.

### Interview Closing Answer

> **“For petabyte-scale reporting, my primary strategy would be partition pruning, columnar storage, clustering/data skipping, and pre-aggregated tables rather than simply adding indexes. I would typically partition by a query-relevant column such as date, avoid high-cardinality partition keys, cluster on frequently filtered or joined columns, compact small files, and create Gold-level aggregates for dashboard workloads. Finally, I would validate the design using query scan volume, execution time, concurrency and cost.”**

### Quick interview flow

**Query Patterns → Date Partitioning → Avoid Over-Partitioning → Columnar Storage → Clustering/Data Skipping → Selective Indexing → File Compaction → Aggregate Tables → Hot/Cold Data → Monitor Query Performance**.


## Round 3 – Managerial / Behavioral

**11.** Tell me about a time you resolved a critical pipeline outage just before a major reporting deadline.

**12.** How do you communicate complex data solutions to risk/compliance teams who are non-technical?

**13.** If a stakeholder wants a new data product on an impossible timeline, how do you handle it?

**14.** Describe a situation where you had to coordinate with a non-Technical business team.


